# 🥘 FoodSight AI - Robust Colab Training (v6 - Final Fix)

**Target:** Train **MobileNetV3Large** on 100 Indian Food Classes  
**Environment:** Google Colab (T4 GPU) - Keras 3 Compatible  

## 🚀 Upgrades in v6 (Fixes `ValueError`):
- **Optimizer Fix:** Added a dummy optimizer step to initialize variables (momentum slots) *before* the training loop. This prevents the `Creating variables on a non-first call` error.
- **MobileNetV3:** Uses the correct model.
- **Normalization Fix:** Uses raw [0, 255] inputs.
- **Keras 3 Ready:** Robust metric handling.
- **Strict Memory Limits:** Caps GPU usage to ~13GB.
- **Deep Data Scan:** Skips corrupt files.
- **Gradient Accumulation:** Simulates Batch Size = 56 for stable convergence.

In [ ]:
# 1. MOUNT DRIVES & SETUP
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
from tensorflow.keras import layers, models
import os
import shutil
import numpy as np
import gc
import zipfile

# --- CRITICAL: STRICT MEMORY LIMITS (PREVENTS OOM) ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            # Limit to 13GB (leave 3GB headroom for system/display)
            tf.config.set_logical_device_configuration(
                gpu,
                [tf.config.LogicalDeviceConfiguration(memory_limit=13312)]
            )
        print("[SYSTEM] GPU Memory Limited to 13GB (Safe Mode)")
    except RuntimeError as e:
        print(e)

# --- CONFIG ---
DRIVE_ROOT = "/content/drive/MyDrive/FoodSight_Project"  # Adjust path if needed
DATASET_ZIP = os.path.join(DRIVE_ROOT, "foodsight_dataset.zip")
LOCAL_DATASET_DIR = "/content/dataset"
RESULTS_DIR = os.path.join(DRIVE_ROOT, "Training_Results_v6_Final")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"[CONFIG] Results will save to: {RESULTS_DIR}")

In [ ]:
# 2. EXTRACT DATASET
if not os.path.exists(LOCAL_DATASET_DIR):
    print(f"[EXTRACT] Unzipping dataset... This takes 2-3 mins.")
    if not os.path.exists(DATASET_ZIP):
        raise FileNotFoundError(f"Dataset zip not found at {DATASET_ZIP}")
        
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATASET_DIR)
    print("[DONE] Dataset Extracted")
else:
    print("[SKIP] Dataset already extracted")

# Find actual data root
INPUT_PATH = LOCAL_DATASET_DIR
for root, dirs, files in os.walk(LOCAL_DATASET_DIR):
    if 'train' in dirs and 'validation' in dirs:
        INPUT_PATH = root
        break
print(f"[OK] Training Root: {INPUT_PATH}")

In [ ]:
# 3. DEEP DATA SCAN (MANDATORY)
# Identifies corrupted images that crash training

print("[SCAN] Starting Deep Scan... (approx 2 mins)")
corrupted_files = set()

def test_tf_decode(filepath):
    try:
        img_raw = tf.io.read_file(filepath)
        img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
        img.numpy()  # Force decode
        return True
    except:
        return False

for subset in ['train', 'validation']:
    subset_path = os.path.join(INPUT_PATH, subset)
    count = 0
    for root, dirs, files in os.walk(subset_path):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                path = os.path.join(root, f)
                # 1. Check size
                if os.path.getsize(path) == 0:
                    corrupted_files.add(path)
                # 2. Check decode
                elif not test_tf_decode(path):
                    corrupted_files.add(path)
                
                count += 1
                if count % 2000 == 0: print(f"  Scanned {count} images in {subset}...")

print(f"[SCAN COMPLETE] Found {len(corrupted_files)} corrupted files. They will be skipped.")

In [ ]:
# 4. ADVANCED DATA LOADERS (Mixup + Skip Corrupt + Raw Inputs)

BATCH_SIZE = 14       # Physical batch (fits in GPU)
ACCUM_STEPS = 4       # Gradient Accumulation steps
EFFECTIVE_BATCH = BATCH_SIZE * ACCUM_STEPS  # = 56

def mixup(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    weight = tf.random.uniform([], 0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = weight * images + (1 - weight) * tf.gather(images, indices)
    mixed_labels = weight * labels + (1 - weight) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

def create_dataset(directory, subset_name, use_mixup=False):
    print(f"[LOAD] {subset_name}...")
    class_names = sorted([d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))])
    num_classes = len(class_names)
    
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            fp = os.path.join(cls_dir, f)
            if fp not in corrupted_files and f.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(fp)
                labels.append(idx)
                
    print(f"  Loaded {len(paths)} valid images in {num_classes} classes")
    
    def load_img(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3]) 
        img = tf.image.resize(img, (224, 224))
        
        # --- CRITICAL FIX: DO NOT DIVIDE BY 255.0 FOR MOBILENETV3 ---
        # MobileNetV3 builds 'Rescaling' layer internally. Expects inputs [0, 255].
        img = tf.cast(img, tf.float32) 
        # img = img / 255.0  <-- REMOVED! This was causing issues.
        
        return img, tf.one_hot(label, num_classes)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if use_mixup:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=True)
    
    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    
    if use_mixup:
        ds = ds.map(lambda x,y: mixup(x,y), num_parallel_calls=tf.data.AUTOTUNE)
        
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds, num_classes, class_names, len(paths)

train_ds, NUM_CLASSES, CLASS_NAMES, TRAIN_IMG_COUNT = create_dataset(os.path.join(INPUT_PATH, "train"), "TRAIN", use_mixup=True)
val_ds, _, _,_ = create_dataset(os.path.join(INPUT_PATH, "validation"), "VAL", use_mixup=False)

In [ ]:
# 5. MODEL SETUP (MobileNetV3Large)
print("[MODEL] Using MobileNetV3Large (Pretrained)...")
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3), 
    include_top=False, 
    weights='imagenet',
    minimalistic=False,
    include_preprocessing=True  # Important! Handles scaling.
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
# Augmentation inside model
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.12)(x)
x = layers.RandomZoom(0.1)(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
print("[MODEL] Ready")

In [ ]:
# 6. CUSTOM TRAINING LOOP (Gradient Accumulation)
# Robust against Keras versions (v2/v3) and OOM

EPOCHS = 20
start_lr = 0.001

# Optimizer & Loss
optimizer = tf.keras.optimizers.Adam(learning_rate=start_lr)
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
acc_metric = tf.keras.metrics.CategoricalAccuracy()
val_acc_metric = tf.keras.metrics.CategoricalAccuracy()

# Accumulators
grad_accum = [tf.Variable(tf.zeros_like(v), trainable=False) for v in model.trainable_variables]

# --- FIX: FORCE INITIALIZE OPTIMIZER VARIABLES ---
# This creates the optimizer slots (m, v) BEFORE the tf.function runs.
# Prevents 'Creating variables on a non-first call' error.
print("[SETUP] Initializing optimizer slots... (prevents TF error)")
zero_grads = [tf.zeros_like(w) for w in model.trainable_variables]
optimizer.apply_gradients(zip(zero_grads, model.trainable_variables))
print("[OK] Optimizer Initialized")

@tf.function
def train_step(x, y, accum_step):
    with tf.GradientTape() as tape:
        preds = model(x, training=True)
        loss = loss_fn(y, preds) / ACCUM_STEPS
    
    grads = tape.gradient(loss, model.trainable_variables)
    for i, g in enumerate(grads):
        grad_accum[i].assign_add(g)
        
    if accum_step == ACCUM_STEPS - 1:
        optimizer.apply_gradients(zip(grad_accum, model.trainable_variables))
        for v in grad_accum: v.assign(tf.zeros_like(v))
        
    acc_metric.update_state(y, preds)
    return loss * ACCUM_STEPS

@tf.function
def val_step(x, y):
    val_acc_metric.update_state(y, model(x, training=False))

# --- KERAS VERSION COMPATIBILITY HELPER ---
# Handles reset_states vs reset_state
def reset_metric_safe(m):
    if hasattr(m, 'reset_states'):
        m.reset_states()
    elif hasattr(m, 'reset_state'):
        m.reset_state()

best_acc = 0.0
print(f"[TRAIN] Starting 20 Epochs with Effective Batch {EFFECTIVE_BATCH}...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # Safe reset for Keras 2/3
    reset_metric_safe(acc_metric)
    reset_metric_safe(val_acc_metric)
    
    # Training
    for step, (x, y) in enumerate(train_ds):
        loss = train_step(x, y, step % ACCUM_STEPS)
        if step % 100 == 0: print(f".", end="")
    
    # Validation
    for x, y in val_ds: val_step(x, y)
    
    # Cast TF scalar to float explicitly for formatting
    train_acc = float(acc_metric.result())
    val_acc = float(val_acc_metric.result())
    
    print(f" -> Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    
    # Save Best
    if val_acc > best_acc:
        best_acc = val_acc
        model.save(os.path.join(RESULTS_DIR, "best_model_v6_final.keras"))
        print("  [SAVED] New Best Model")
    
    gc.collect()  # Anti-OOM

In [ ]:
# 7. FINE TUNING (Optional)
# Unfreeze top layers safely
print("[FINE TUNE] Unfreezing top 30 layers...")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

optimizer.learning_rate.assign(1e-5)  # Very low LR

# Re-init accumulators for new variables
grad_accum = [tf.Variable(tf.zeros_like(v), trainable=False) for v in model.trainable_variables]

print("Run cell 6 again to continue training if needed!")